# Gaussian Naive Bayes

<div style="float: right; margin-left: 10px; max-width: 150px;">
    <img alt="Iris Blume" src="https://openclipart.org/download/297883/BrownFloweredIris.svg" width="140px" />
</div>

In diesem Notebook experimentieren wir mit der Modellfamilie *Gaussian Naive Bayes* und vergleichen den Klassifikator mit dem logistischen Regressionsmodell.

Darüber hinaus gibt es zwei neuartige Themen:

1. Wie man mit gzip-Dateien arbeitet;
2. Was ist Kreuzvalidierung und wie wird sie verwendet?

### Datensatz

Wir werde den [Iris-Datensatz](https://archive.ics.uci.edu/dataset/53/iris) verwenden. Der Datensatz enthält 3 Klassen mit jeweils 50 Instanzen, wobei sich jede Klasse auf eine Art Irispflanze ([Schwertlilie](https://de.wikipedia.org/wiki/Schwertlilien)) bezieht. Es enthält 4 reele Merkmale: Kelchblattlänge, Kelchblattbreite, Blütenblattlänge und Blütenblattbreite.

## Benötigte Module

In [ ]:
import csv
import gzip
import numpy as np
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import os

## Zur Erinnerung: Naive Bayes (kategorische Features)

Wir erinnern uns an den Satz von Bayes:

$$P(y \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid y)\, P(y)}{P(\mathbf{x})}$$

Unter der **Naive-Bayes-Annahme** (bedingte Unabhängigkeit der Features gegeben die Klasse) gilt:

$$P(\mathbf{x} \mid y) = \prod_{j=1}^{d} P(x_j \mid y)$$

Bei kategorischen Features haben wir $P(x_j \mid y=c)$ direkt aus den relativen Häufigkeiten geschätzt.

### Was, wenn die Features *kontinuierlich* sind?

Idee: Wir nehmen an, dass jedes Feature $x_j$ innerhalb einer Klasse $c$ **normalverteilt** ist:

$$P(x_j \mid y=c) = \frac{1}{\sqrt{2\pi\,\sigma_{jc}^2}} \exp\!\left(-\frac{(x_j - \mu_{jc})^2}{2\,\sigma_{jc}^2}\right)$$

Die Parameter $\mu_{jc}$ und $\sigma_{jc}^2$ schätzen wir aus den Trainingsdaten (Stichprobenmittelwert und -varianz der Klasse $c$ für Feature $j$).

Das ist der **Gaussian Naive Bayes**-Klassifikator.

## Gzip-komprimierte Dateien laden

Bisher haben wir CSV-Dateien im Klartext gelesen. In der Praxis sind Datensätze oft komprimiert gespeichert (z.B. als `.csv.gz`). Das Python-Modul `gzip` erlaubt es, solche Dateien direkt zu öffnen, ohne sie vorher manuell zu entpacken.

### Übung: CSV-Datei aus einer `.gz`-Datei laden

Implementieren Sie die Funktion `load_csv_gz`. Sie soll:
1. Die Datei mit [`gzip.open()`](https://docs.python.org/3/library/gzip.html#gzip.open) im Textmodus öffnen.
2. Die Zeilen mit [`csv.reader()`](https://docs.python.org/3/library/csv.html#csv.reader) lesen.
3. Die erste Zeile als Header speichern.
4. Die restlichen Zeilen in Features (Spalten 0–3, als `float`) und Labels (Spalte 4, als `str`) aufteilen.

**Rückgabe:** `(x, y, feature_names, species)` wobei `x` ein `np.ndarray` mit Shape `(n, 4)` und `y` ein `np.ndarray` mit Strings ist.

In [ ]:
def load_iris(path_csv: str) \
    -> tuple[np.ndarray, np.ndarray, list[str], dict]:
    """
    Ladet den Iris-Datensatz aus einer gzip-komprimierte CSV-Datei.

    :param path_csv: Pfad zur `.csv.gz`-Datei.
    :return: Tupel der Form `(x, y)`, wobei
      - `x`: Merkmalsmatrix als Array der Form `(sample_count, feature_count)`.
      - `y`: Ausgabevektor als Array der Form `(sample_count)`; Elemente sind `str`.
      - `feature_names` die Spaltennamen aus der ersten Zeile beinhaltet;
      - `species` Mapping von Artennamen zu Farben ist;
    """
    species = {'setosa': '#E41A1C', 'versicolor': '#377EB8', 'virginica': '#4DAF4A'}
    x, y, feature_names = [], [], []
    
    pass

    # Listen mit Zahlen zu Numpy-Arrays konvertieren
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.uint16)
    return x, y, feature_names, species


# Daten laden und prüfen; bitte Pfad anpassen
x, y, feature_names, species = load_iris('iris.csv.gz')

print(f'Merkmale: {feature_names}')
print(f' x.shape: {x.shape}')
print(f' y.shape: {y.shape}')
print(f' Klassen: {species.keys()}')
print('\nErste 5 Zeilen von X:')
print(x[:5, :])

## 3. Datenexploration und Visualisierung

Bevor wir ein Modell trainieren, schauen wir uns die Daten an. Wir plotten die Features gegeneinander und prüfen visuell, ob die Gausssche Annahme (Normalverteilung pro Klasse) plausibel ist.

### Übung: Streudiagramm (Scatter Plot)

Implementieren Sie die Funktion zur Erzeugung eines Streudiagramms. Färben Sie die Punkte nach Klasse ein. Diese Funktion wird verwendet, um  **Blütenblattlänge** (Feature-Index 2, x-Achse) gegen **Blütenblattbreite** (Feature-Index 3, y-Achse) zu visualisieren.

In [ ]:
def plot_iris_scatter(x: np.ndarray, y: np.ndarray, feature_names: list[str],
                      species: dict[str, str], feature_x: int, feature_y: int):
    figure, ax = plt.subplots(figsize=(6, 4), dpi=100)

    pass

    return figure


_ = plot_iris_scatter(x, y, feature_names, species, 2, 3)
plt.show()

### Histogramme pro Feature und Klasse mit Gauss-Überlagerung

Hier überprüfen wir visuell, ob die Normalverteilungsannahme pro Klasse sinnvoll ist.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(10, 7), dpi=100)
axes = axes.ravel()

for j, ax in enumerate(axes):
    for k, (class_name, class_color) in enumerate(species.items()):
        values = x[y == k, j]
        mu, sigma = values.mean(), values.std()

        # Histogramm (normiert, also Dichte)
        ax.hist(values, bins=12, density=True, alpha=0.35, color=class_color, label=class_name)

        # Gauss-Kurve überlagern
        xs = np.linspace(values.min() - 0.5, values.max() + 0.5, 200)
        pdf = (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * ((xs - mu) / sigma) ** 2)
        ax.plot(xs, pdf, color=class_color, linewidth=2)

    ax.set(axisbelow=True, title=feature_names[j])
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Diskussion:** Schauen Sie sich die Histogramme an.
- Für welche Features / Klassen passt die Normalverteilung gut?
- Wo sieht man Abweichungen?
- Erwarten Sie, dass Gaussian Naive Bayes hier gut funktioniert?

## Gaussian Naive Bayes

Wir implementieren den Klassifikator in vier Schritten:

| Schritt | Was? | Formel |
|---------|------|--------|
| 1 | Prior $P(y=c)$ | $\frac{n_c}{n}$ |
| 2 | Klassenstatistiken $\mu_{jc},\; \sigma_{jc}^2$ | Stichprobenmittelwert / -varianz |
| 3 | Log-Likelihood einer Beobachtung | $\log P(x_j \mid y=c) = -\frac{1}{2}\log(2\pi\sigma_{jc}^2) - \frac{(x_j - \mu_{jc})^2}{2\sigma_{jc}^2}$ |
| 4 | Vorhersage | $\hat{y} = \arg\max_c \left[\log P(y=c) + \sum_j \log P(x_j \mid y=c)\right]$ |

Wir rechnen im **Log-Raum**, um numerische Unterläufe bei der Multiplikation vieler kleiner Wahrscheinlichkeiten zu vermeiden (dieselbe Idee wie bei der Log-Likelihood aus der Vorlesung).

### Schritt 1: A-priori-Wahrscheinlichkeiten (Priors)

$$P(y = c) = \frac{\text{Anzahl Instanzen der Klasse } c}{\text{Gesamtzahl Instanzen}}$$

### Übung: Priors berechnen

Tipp: Wir haben schon mit der Funktion [`np.bincount`](https://numpy.org/doc/2.1/reference/generated/numpy.bincount.html) Erfahrung.

In [ ]:
def compute_priors(y: np.ndarray) -> np.ndarray:
    """
    Berechnet die A-priori-Wahrscheinlichkeit jeder Klasse.

    :param y: Array der Klassenindizes.
    :return: Log-Priors, also log P(y=c) für jede Klasse c
    """
    pass


log_priors = compute_priors(y)
for class_name, lp in zip(species.keys(), log_priors):
    print(f'P(y={class_name:10s}) = {np.exp(lp):.4f}  (log = {lp:.4f})')

### Schritt 2 – Klassenstatistiken (Mittelwert und Varianz)

Für jede Klasse $c$ und jedes Feature $j$ berechnen wir:

$$\mu_{jc} = \frac{1}{n_c} \sum_{i:\, y_i = c} x_{ij} \qquad\qquad
\sigma_{jc}^2 = \frac{1}{n_c} \sum_{i:\, y_i = c} (x_{ij} - \mu_{jc})^2$$

> *Hinweis: Wir verwenden hier die populationsbasierte Varianz (Teiler $n_c$), nicht die Stichprobenvarianz (Teiler $n_c - 1$). Beides ist üblich; der Unterschied ist bei genügend Daten vernachlässigbar.*

### Übung: Mittelwerte und Varianzen berechnen

In [ ]:
def compute_class_statistics(x, y):
    """
    Berechnet pro Klasse den Mittelwert und die Varianz jedes Features.

    Parameter:
    ----------
    :param x: Feature-Matrix als Array der Form `(n, d)`.
    :param y: Klassenindizes als Array der Form `(n,)`.
    :return: Tupel `(mu, variance)`, wobei
      - `mu` die Mittelwerte pro Klasse und Feature ist.
      - `variance` die Varianzen pro Klasse und Feature ist.
      Beide sind Arrays der Form `(K, d)`
    """
    K, d = y.max().item() + 1, x.shape[1]
    mu, variance  = np.zeros((K, d)), np.zeros((K, d))

    pass

    return mu, variance


mu, variance = compute_class_statistics(x, y)
print('Mittelwerte (mu):')
print(mu)
print('\nVarianzen (variance):')
print(variance)

### Schritt 3 – Gauss'sche Log-Likelihood

Für einen einzelnen Wert $x$ bei gegebenem $\mu$ und $\sigma^2$:

$$\log P(x \mid \mu, \sigma^2) = -\frac{1}{2}\,\log(2\pi\,\sigma^2) \;-\; \frac{(x - \mu)^2}{2\,\sigma^2}$$

### Übung: Log-PDF der Normalverteilung

Tipp: Die Konstante `np.pi` ist der Wert von $\pi$.

In [ ]:
def gauss_log_pdf(x, mu, var):
    """
    Berechnet die Log-Wahrscheinlichkeitsdichte der Normalverteilung.

    Parameter:
    ----------
    :param x: Beobachteter Wert oder Werte.
    :param mu: Mittelwert(e) der Verteilung.
    :param var: Varianz(en) der Verteilung.
    :return: log N(x | mu, var)
    """
    pass


# Schnelltest: Standardnormalverteilung bei x=0 → log-PDF ≈ -0.9189
print(f'gauss_log_pdf(0, 0, 1) = {gauss_log_pdf(0, 0, 1):.4f}')
print(f'Erwartet:                -0.9189')

### Schritt 4 – Vorhersage

Für eine neue Beobachtung $\mathbf{x} = (x_1, \dots, x_d)$ berechnen wir den
**log-Posterior** für jede Klasse:

$$\log P(y=c \mid \mathbf{x}) \;\propto\; \underbrace{\log P(y=c)}_{\text{log-Prior}} \;+\; \sum_{j=1}^{d} \underbrace{\log P(x_j \mid y=c)}_{\text{Gauss log-PDF}}$$

Die vorhergesagte Klasse ist die mit dem höchsten log-Posterior.

### Übung: Vorhersagefunktion

Implementieren Sie die Vorhersagefunktion basierend auf die Definition des Log-Posteriors.

In [ ]:
def nb_prediction(x: np.ndarray, log_priors: np.ndarray,
                  mu: np.ndarray, variance: np.ndarray) -> np.ndarray:
    """
    Sagt für jede Instanz in X_neu die Klasse vorher (Gaussian Naive Bayes).

    :param x: Array der Form `(m, d)` mit Instanzen, für die eine Vorhersage
              getroffen werden soll.
    :param log_priors: Log-Prior-Wahrscheinlichkeiten als Array der Form `(K,)`.
    :param mu: Mittelwerte pro Klasse und Feature als Array der Form `(K, d)`.
    :param variance: Varianzen pro Klasse und Feature als Array der Form `(K, d)`.
    :return: Vorhergesagte Klassenindizes als Array der Form `(m,)`.
    """
    m, K = x.shape[0], mu.shape[0]
    log_posterior = np.zeros((m, K), dtype=np.float64)

    # Für jede Klasse k: log_posterior[:, k] = log_prior[k] + Summe der gauss_log_pdf
    #                     über alle Features j
    pass


# Schnelltest: Vorhersage auf den gesamten Trainingsdaten
y_hat = nb_prediction(x, log_priors, mu, variance)
accuracy = np.mean(y_hat == y)
print(f'Trainingsgenauigkeit für alle Daten: {accuracy:.2%}')

### Stift-und-Papier-Übung

Betrachte den folgenden Testpunkt mit **zwei Features** (Blütenblattlänge, Blütenblattbreite)
und **zwei Klassen** (versicolor, virginica).

| | Blütenblattlänge | Blütenblattbreite |
|---|---|---|
| Testpunkt $\mathbf{x}^*$ | **5.0** | **1.8** |

Gegebene Parameter (aus den Daten geschätzt):

| Klasse | $\mu_1$ | $\sigma_1^2$ | $\mu_2$ | $\sigma_2^2$ | Prior |
|--------|---------|-------------|---------|-------------|-------|
| versicolor | 4.26 | 0.22 | 1.33 | 0.04 | 0.5 |
| virginica  | 5.55 | 0.30 | 2.03 | 0.07 | 0.5 |

Berechnen Sie per Hand den Log-Posterior für beide Klassen und bestimmen Sie die vorhergesagte Klasse.

*Hinweis: $\log(2\pi) \approx 1.8379$*

<details>
<summary><b>Lösung aufklappen</b></summary>

**Versicolor:**

$\log P(x_1 \mid \text{versi}) = -\frac{1}{2}\log(2\pi \cdot 0.22) - \frac{(5.0 - 4.26)^2}{2 \cdot 0.22} = -\frac{1}{2}(1.8379 + \log 0.22) - \frac{0.5476}{0.44}$

$= -\frac{1}{2}(1.8379 - 1.5141) - 1.2445 = -0.1619 - 1.2445 = -1.4064$

$\log P(x_2 \mid \text{versi}) = -\frac{1}{2}\log(2\pi \cdot 0.04) - \frac{(1.8 - 1.33)^2}{2 \cdot 0.04}$

$= -\frac{1}{2}(1.8379 - 3.2189) - \frac{0.2209}{0.08} = 0.6905 - 2.7613 = -2.0708$

Log-Posterior: $\log(0.5) + (-1.4064) + (-2.0708) = -0.6931 - 1.4064 - 2.0708 = -4.1703$

**Virginica:**

$\log P(x_1 \mid \text{virg}) = -\frac{1}{2}\log(2\pi \cdot 0.30) - \frac{(5.0 - 5.55)^2}{2 \cdot 0.30}$

$= -\frac{1}{2}(1.8379 - 1.2040) - \frac{0.3025}{0.60} = -0.3170 - 0.5042 = -0.8212$

$\log P(x_2 \mid \text{virg}) = -\frac{1}{2}\log(2\pi \cdot 0.07) - \frac{(1.8 - 2.03)^2}{2 \cdot 0.07}$

$= -\frac{1}{2}(1.8379 - 2.6593) - \frac{0.0529}{0.14} = 0.4107 - 0.3779 = 0.0328$

Log-Posterior: $\log(0.5) + (-0.8212) + 0.0328 = -0.6931 - 0.8212 + 0.0328 = -1.4815$

**Ergebnis:** $-1.4815 > -4.1703$ → **virginica** gewinnt ✓

</details>

## Train/Test-Split und Auswertung

Wie wir in der Vorlesung gesehen haben, bewerten wir ein Modell nicht auf den Trainingsdaten, sondern auf einem separaten Testset.

In [ ]:
def train_test_split(x, y, test_fraction=0.3):
    """
    Teilt die Daten zufällig in Trainings- und Testsatz auf.

    :param x: Feature-Matrix als Array der Form `(n, d)`.
    :param y: Klassenindizes als Array der Form `(n,)`.
    :param test_fraction: Anteil der Daten für das Testset (zwischen 0 und 1).
    :return: Tupel `(x_train, y_train, x_test, y_test)`.
    """
    generator = np.random.default_rng(19751979)
    idx = generator.permutation(y.size)
    split = int(y.size * (1 - test_fraction))
    return x[idx[:split]], y[idx[:split]], x[idx[split:]], y[idx[split:]]


x_train, y_train, x_test, y_test = train_test_split(x, y, test_fraction=0.25)
print(f'Trainingsdaten: {x_train.shape[0]} Instanzen')
print(f'     Testdaten: {x_test.shape[0]} Instanzen')

In [ ]:
# Modell auf Trainingsdaten trainieren
log_priors_tr = compute_priors(y_train)
mu_tr, var_tr = compute_class_statistics(x_train, y_train)

# Vorhersage auf Testdaten
y_hat_test = nb_prediction(x_test, log_priors_tr, mu_tr, var_tr)
print(f'Testgenauigkeit: {np.mean(y_hat_test == y_test):.2%}')

### Übung: Konfusionsmatrix

Implementieren Sie eine Funktion, die eine Konfusionsmatrix aus wahren und vorhergesagten Labels erstellt. Die Zeilen entsprechen den **wahren** Klassen, die Spalten den **vorhergesagten** Klassen.

In [ ]:
def confusion_matrix(y_true: np.ndarray, y_predicted: np.ndarray) -> np.ndarray:
    """
    Erstellt die Konfusionsmatrix.

    :param y_true: Wahre Klassenindizes als Array der Form `(m,)`.
    :param y_predicted: Vorhergesagte Klassenindizes für dieselbe Beobachtungen als `y_true`.
    :return: Konfusionsmatrix als Array der Form `(K, K)`. Element `[i, j]`
             speichert die Anzahl Instanzen mit wahrer Klasse `i` und
             vorhergesagter Klasse `j`.
    """
    K = max(y_true.max().item(), y_predicted.max().item()) + 1
    result = np.zeros((K, K), dtype=np.uint32)

    pass

    return result

Der folgende Codeabschnitt visualisiert die Konfusionsmatrix in einem Plot.

In [ ]:
cm = confusion_matrix(y_test, y_hat_test)
class_names = list(species.keys())

_, ax = plt.subplots(figsize=(5, 4), dpi=100)
im = ax.imshow(cm, cmap='Blues')
ax.set(xticks=range(len(class_names)), yticks=range(len(class_names)))
ax.set(xticklabels=class_names, yticklabels=class_names)
ax.set(xlabel='Vorhergesagte Klasse', ylabel='Wahre Klasse')

# Zahlenwerte in die Zellen schreiben
for i in range(len(class_names)):
    for j in range(len(class_names)):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=16)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Mit der Hilfe der Konfusionsmatrix können wir viele Metriken berechnen.

In [ ]:
print(f"{'Klasse':>11s}  {'Precision':>9s}  {'Recall':>6s}  {'F1':>6s}")
print('-----------------------------------------')
for i, c in enumerate(species.keys()):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp    # andere Klassen, die als c vorhergesagt wurden
    fn = cm[i, :].sum() - tp    # Klasse c, die als andere vorhergesagt wurde
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    if (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0
    print(f'{c:>11s} {precision:>10.2%}  {recall:>7.2%}  {f1:>7.2%}')

print(f'\nGesamtgenauigkeit (Accuracy): {np.trace(cm) / cm.sum():.2%}')

### Welche Klasse?

Im folgenden Plot wird ein zufälliger Testpunkt hervorgehoben. **Bevor Sie die nächste Zelle ausführen,** raten Sie, welche Klasse der Naive-Bayes-Klassifikator vorhersagen wird!

In [ ]:
# Einen zufälligen Testpunkt auswählen
generator = np.random.default_rng(19751979)
i_drawn = generator.choice(x_test.shape[1])
x_drawn = x_test[i_drawn, :]

fig, ax = plt.subplots()
for index_class, (class_name, class_color) in enumerate(species.items()):
    mask = y_train == index_class
    ax.scatter(x_train[mask, 2], x_train[mask, 3],
               c=class_color, label=class_name, alpha=0.4, edgecolors='w', s=40)

# Testpunkt hervorheben
ax.scatter(x_drawn[2], x_drawn[3], c='black', marker='x', s=150, zorder=5,
           label='Testpunkt')
ax.set(axisbelow=True, xlabel=feature_names[2], ylabel=feature_names[3])
ax.set(title='Welche Klasse hat der schwarze Punkt?')
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Auflösung

class_names = list(species.keys())
y_hat = nb_prediction(x_drawn.reshape(1, -1), log_priors_tr, mu_tr, var_tr)[0]
print(f'  Vorhersage: {class_names[y_hat]}')
print(f'Wahre Klasse: {class_names[y_test[i_drawn]]}')

## Kreuzvalidierung (Cross-Validation)

### Motivation

Beim einfachen Train/Test-Split hängt das Ergebnis davon ab, *welche* Datenpunkte zufällig im Test- bzw. Trainingsset landen. Das macht die Bewertung **instabil** – ein anderer Seed gibt eine andere Genauigkeit.

[**Kreuzvalidierung**](https://de.wikipedia.org/wiki/Kreuzvalidierungsverfahren) (Cross-Validation) löst dieses Problem, indem *jede* Instanz genau einmal als Testdatum verwendet wird.

<img alt="Kreuzvalidierung" width="400px" src="https://upload.wikimedia.org/wikipedia/commons/b/b5/K-fold_cross_validation_EN.svg" />

### k-Fold Cross-Validation – Ablauf

1. Teile die $n$ Datenpunkte zufällig in $k$ gleich große **Folds** auf.
2. Für jede Runde $i = 1, \dots, k$:
   - Fold $i$ ist das **Testset**.
   - Alle anderen Folds zusammen sind das **Trainingsset**.
   - Trainiere das Modell und berechne die Genauigkeit auf Fold $i$.
3. Berichte den **Mittelwert** und die **Standardabweichung** der $k$ Genauigkeiten.

<span style="color: grey; font-size: 0.5em;">Abbildung von Wikimedia Commons; Copyright [Gufosowa](https://en.wikipedia.org/wiki/File:K-fold_cross_validation_EN.svg).</span>

### Visualisierung der k-Fold-Aufteilung (am Beispiel k=5)

In [ ]:
def plot_cv(k: int):
    figure, ax = plt.subplots(figsize=(9, 3), dpi=100)
    for fold in range(k):
        for block in range(k):
            if block == fold:
                color = '#E41A1C'
                label_text = 'Test'
            else:
                color = '#377EB8'
                label_text = 'Train' if fold == 0 and block == 1 else None
            ax.barh(fold, 1, left=block, color=color, edgecolor='white', height=0.7, label=label_text)
    ax.set(xlim=[0, k], xticks=np.arange(k) + 0.5, yticks=range(k))
    ax.set_yticklabels([f'Round {i + 1}' for i in range(k)])
    ax.set_xticklabels([f'Fold {i + 1}' for i in range(k)])
    ax.set_title(f'{k}-Fold Cross-Validation')
    ax.invert_yaxis()
    return figure


plot_cv(5)
plt.tight_layout()
plt.show()

### Übung: k-Fold Kreuzvalidierung implementieren

Implementieren Sie die Funktion `cross_validation`. Sie soll:
1. Die Indizes zufällig mischen.
2. Die Indizes in $k$ Folds aufteilen (Tipp: `np.array_split`).
3. In jeder Runde: Trainingsset und Testset bestimmen, Naive Bayes trainieren, Genauigkeit auf dem Testfold berechnen.
4. Alle $k$ Genauigkeiten als Array zurückgeben.

In [ ]:
def cross_validation(x: np.ndarray, y: np.ndarray, k: int = 5) -> np.ndarray:
    """
    Führt k-Fold Kreuzvalidierung mit Gaussian Naive Bayes durch.

    :param x: Feature-Matrix als Array der Form `(n, d)`.
    :param y: Klassenindizes als Array der Form `(n,)`.
    :param k: Anzahl der Folds.
    :return: Array der Form `(n,)` mit den Vorhersagen für alle `n` Beobachtungen.
    """
    generator = np.random.default_rng(19751979)
    indices = generator.permutation(y.size)

    result = np.empty(y.size)

    pass
    # folds definieren
    # Für jeden Fold i:
    #   - test_idx  = folds[i]
    #   - train_idx = alle anderen Folds zusammen (np.concatenate)
    #   - Trainiere Naive Bayes (berechne_priors, berechne_klassenstatistiken)
    #   - Vorhersagen auf Testfold (nb_vorhersage) speichern

    return result


for k in [5, 10]:
    y_hat = cross_validation(x, y, k)
    print(f'{k:2d}-Fold CV Genauigkeit: {np.mean(y_hat == y):.1%}')

## Entscheidungsgrenzen: Naive Bayes vs. Logistische Regression

Zum Abschluss vergleichen wir die **Entscheidungsgrenzen** beider Klassifikatoren. Dazu verwenden wir nur zwei Features (Blütenblattlänge und Blütenblattbreite), damit wir die Grenzen in 2D visualisieren können.

Die logistische Regression haben wir bereits in einem früheren Notebook implementiert. Hier stellen wir eine kompakte Version bereit mit der Softmax-Variante für Mehrklassenprobleme.

In [ ]:
def add_bias(x: np.ndarray) -> np.ndarray:
    """
    Fügt Bias-Spalte zur Merkmal-Matrix hinzu.

    :param x: Merkmalsmatrix mit `n` Beobachtungen als Array der Form `(n, d)`.
    :return: Array der Form `(n, d + 1)` in dem die erste Spalte nur Eines beinhaltet.
    """
    bias = np.ones((x.shape[0], 1), dtype=x.dtype)
    return np.hstack((bias, x))


def softmax(z: np.ndarray) -> np.ndarray:
    """
    Berechnet die Softmax-Funktion zeilenweise (numerisch stabil).

    :param z: Matrix der linearen Scores als Array der Form `(m, K)`.
    :return: Wahrscheinlichkeitsmatrix mit derselben Form als `z`; jede Zeile
             summiert sich zu 1.
    """
    exp_z = np.exp(z - z.max(axis=1, keepdims=True))
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def train_logistic_regression(x: np.ndarray, y: np.ndarray, learning_rate: float = 0.1,
                              epoch_count: int = 500):
    """
    Trainiert logistische Regression mit Gradientenabstieg.

    :param x: normalisierte Markmalswerte als Array der Form `(n, d)`.
    :param y: Vektor von Klassenindizes als Array der Form `(n,)`.
    :param learning_rate: Lernrate (Schrittgrösse).
    :param epoch_count: Anzahl der Iterationen.
    :return: Modellparameter (Gewichtsmatrix) als Array der Form `(d + 1, K)`.
    """
    # One-Hot-Encoding der Labels
    K = y.max().item() + 1
    y_one_hot = np.zeros((x.shape[0], K), dtype=np.float64)
    for k in range(K):
        y_one_hot[y == k, k] = 1.0

    # Bias-Spalte (Einsen) hinzufügen
    x = add_bias(x)

    # Gewichte initialisieren
    result = np.zeros((x.shape[1], K))
    for _ in range(epoch_count):
        p = softmax(x @ result)  # Vorhersage-Wahrscheinlichkeiten
        gradient = x.T @ (p - y_one_hot) / x.shape[0]
        result -= learning_rate * gradient

    return result


def lr_prediction(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    """
    Sagt Klassen mit einer trainierten logistischen Regression vorher.

    :param x: Feature-Matrix als Array der Form `(m, d)`.
    :param w: Modellparameter als Array der Form `(d + 1, K)`.
    :return: Vorhergesagte Klassenindizes als Array der Form `(m,)`.
    """
    probabilities = softmax(add_bias(x) @ w)
    return np.argmax(probabilities, axis=1)

### Entscheidungsgrenzen plotten

Wir trainieren beide Modelle auf den gleichen zwei Features und zeichnen die
Entscheidungsgrenzen als farbige Hintergrundregionen.

In [ ]:
# Nur zwei Features verwenden: Blütenblattlänge (2) und Blütenblattbreite (3)
feature_indices = [2, 3]
x_selected = x[:, feature_indices]

# Naive Bayes trainieren
priors = compute_priors(y)
mu, variance = compute_class_statistics(x_selected, y)

# Logistische Regression trainieren
w_selected = train_logistic_regression(
    x_selected, y, learning_rate=0.5, epoch_count=1000)

# Gitter für die Entscheidungsgrenzen erzeugen
x1_min, x1_max = x_selected[:, 0].min() - 0.5, x_selected[:, 0].max() + 0.5
x2_min, x2_max = x_selected[:, 1].min() - 0.5, x_selected[:, 1].max() + 0.5
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                        np.linspace(x2_min, x2_max, 300))
grid = np.c_[xx1.ravel(), xx2.ravel()]

# Vorhersagen auf dem Gitter
y_nb_grid = nb_prediction(grid, priors, mu, variance)
y_lr_grid = lr_prediction(grid, w_selected)
y_nb_grid = y_nb_grid.reshape(xx1.shape)
y_lr_grid = y_lr_grid.reshape(xx1.shape)

In [ ]:
# Farben: helle Varianten für den Hintergrund
cmap_bg = ListedColormap(['#FBB4AE', '#B3CDE3', '#CCEBC5'])   # pastellrot, -blau, -grün
cmap_fg = ListedColormap(list(species.values()))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100, sharex='all', sharey='all')
for ax, Z, title in zip(axes, [y_nb_grid, y_lr_grid],
                        ['Gaussian Naive Bayes', 'Logistische Regression']):
    ax.pcolormesh(xx1, xx2, Z, cmap=cmap_bg, shading='auto', alpha=0.6)
    for k, (class_name, class_color) in enumerate(species.items()):
        mask = y == k
        ax.scatter(x_selected[mask, 0], x_selected[mask, 1],
                   c=class_color, label=class_name, edgecolors='w', s=40, alpha=0.9)
    ax.set(title=title, xlabel=feature_names[feature_indices[0]])
    if title == 'Gaussian Naive Bayes':
        ax.set(ylabel=feature_names[feature_indices[1]])
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

### Diskussion

Schauen Sie sich die beiden Plots an und überlegen Sie:

1. Welche Form haben die Grenzen bei Naive Bayes? Welche bei der logistischen Regression? Warum?
2. Welcher Klassifikator ist hier „besser"? Berechnen und vergleichen Sie CV-Ergebnisse.
3. Welches Modell hat mehr Bias? Welches mehr Varianz?
4. Wann könnte Naive Bayes die logistische Regression schlagen?

## Bonus: Feature-Auswahl-Experiment

Was passiert, wenn wir nur *ein einziges Feature* verwenden? Oder alle vier? Vergleichen wir die CV-Genauigkeit für verschiedene Feature-Kombinationen.

In [ ]:
# Experiment: CV-Genauigkeit mit unterschiedlichen Feature-Mengen
feature_sets = {}
for i, feature_name in enumerate(feature_names):
    feature_sets[feature_name] = [i]
feature_sets[os.path.commonprefix(feature_names[:2])] = [0, 1]
feature_sets[os.path.commonprefix(feature_names[2:])] = [2, 3]
feature_sets['alle 4'] = list(range(len(feature_names)))

print(f'{'Feature-Menge':>20s} {'5-Fold CV Accuracy':>20s}')
print('-----------------------------------------')
for name, idx in feature_sets.items():
    y_hat = cross_validation(x[:, idx], y, k=5)
    print(f'{name:>20s}   {(y == y_hat).mean():.2%} ± {(y == y_hat).std():.2%}')